# Feature Selection Pipeline

This notebook reviews feature selection as a progression-aligned, stability-aware model-selection component for the FRDA MRI biomarker pipeline.

**Methods reviewed**

| Method | Role | Objective |
|---|---|---|
| `none` | Full-panel reference | Uses all eligible prespecified MRI features. |
| `progression_univariate` | Progression-aligned filter | Ranks features by annual paired progression effects for V1->V2 and V2->V3 inside the training fold. |
| `progression_mrmr` | Progression-aware redundancy filter | Greedy forward selection combines annual progression relevance with an absolute-correlation redundancy penalty. |
| `sparse_srm` | Embedded sparse SRM-style selector | Uses interval-balanced annual change statistics and ElasticNet-style shrinkage to define non-zero selected coefficients. |
| `mi_visit` / `mml` | Historical comparators | Retained as sensitivity checks; they are not the main supervisor-facing progression selectors. |

No FARS/SARA or healthy controls are used to select feature-selection methods.


In [11]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.adaptive_tuning import feature_rank_table
from src.eval.elbow import (
    DEFAULT_ELBOW_K_VALUES,
    elbow_candidate_grid,
    elbow_nested_candidates,
    evaluate_elbow_feature_counts,
    select_feature_count_elbow,
)
from src.eval.feature_importance import (
    annual_feature_contributions,
    coefficient_importance_table,
    fit_locked_srm_full_data,
    infer_feature_domains,
)
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import clinical_change_effect_sizes, paired_deltas_from_long, probability_positive_change, reference_effect_sizes
from src.eval.model_selection import select_hierarchical_candidate
from src.eval.panel_combinations import best_panel_combination, evaluate_srm_panel_combinations
from src.eval.stability import selected_feature_jaccard
from src.features.panels import (
    DEFAULT_APRIORI_SPINAL_STRUCTURAL_FEATURE,
    a_priori_70_panels,
    a_priori_70_panel_table,
    resolve_a_priori_70_panel,
    spinal_structural_candidate_audit,
)
from src.features.registry import FEATURE_GROUPS
from src.features.selection import feature_domain_coverage, feature_stability_report, feature_set_jaccard_summary
from src.models.srm_global import (
    srm_global_loocv,
    srm_global_nested_loocv,
)
from src.reporting.fold_comparison import (
    fixed_patient_train_test_split,
    fixed_train_inner_cv_fold_plan_table,
    fixed_train_test_clinical_benchmark_table,
    fixed_train_test_single_feature_benchmark_table,
    fold_train_test_clinical_benchmark_table,
    fold_train_test_single_feature_benchmark_table,
)
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
broad_imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]

panel = resolve_a_priori_70_panel(
    pairs_df.columns,
    spinal_structural_feature=DEFAULT_APRIORI_SPINAL_STRUCTURAL_FEATURE,
)
missing = panel.missing_features
if missing:
    raise KeyError(f"A priori 70-feature panel has missing dataset columns: {missing}")
imaging_cols = [c for c in panel.features if c in long_df.columns]
subject_col = "pair_id"
split_group_col = "subject"
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
HOLDOUT_TEST_FRACTION = 0.20
N_BOOT = 100
RANDOM_SEED = DEFAULT_CONFIG.random_state
ELBOW_K_VALUES = (4, 6, 8, 10, 12, 16, 20, 30, 40)
ELBOW_TOLERANCE = 0.03
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows")
print({
    "feature_pool": panel.name,
    "a_priori_features": len(imaging_cols),
    "broad_imaging_reference_features": len(broad_imaging_cols),
    "subject_col": subject_col,
    "split_group_col": split_group_col,
    "cv_n_splits": CV_N_SPLITS,
})
print("A priori 70-feature audit")
display(panel.audit)
a_priori_feature_rank = feature_rank_table(long_df, imaging_cols, subject_col=subject_col, visit_col="visit")
a_priori_feature_rank.to_csv(RESULTS_DIR / "feature_selection_a_priori_feature_rank.csv", index=False)
print("Top per-feature longitudinal Cohen d_z ranks used as selector relevance")
display(a_priori_feature_rank.head(20))

named_panel_audit = pd.concat([
    a_priori_70_panel_table(pairs_df.columns, family="anatomical"),
    a_priori_70_panel_table(pairs_df.columns, family="modality"),
], ignore_index=True)
named_panel_audit.to_csv(RESULTS_DIR / "feature_selection_named_panel_audit.csv", index=False)
print("Named supervisor panel audit")
display(named_panel_audit)

spinal_structural_audit = spinal_structural_candidate_audit(pairs_df.columns)
spinal_structural_audit.to_csv(RESULTS_DIR / "feature_selection_spinal_structural_audit.csv", index=False)
print("Spinal structural feature requiring PG/supervisor confirmation")
display(spinal_structural_audit)


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_pairs_drop3poms.csv: 414 visit rows
{'feature_pool': 'a_priori_70', 'a_priori_features': 70, 'broad_imaging_reference_features': 146, 'subject_col': 'pair_id', 'split_group_col': 'subject', 'cv_n_splits': 5}
A priori 70-feature audit


,feature,group,present,status
0,Cerebellum_WM_CerebNet,structural_brain,True,confirmed
1,Cerebellum_Cortex_CerebNet,structural_brain,True,confirmed
2,SCP,structural_brain,True,confirmed
3,Medulla,structural_brain,True,confirmed
4,Pons,structural_brain,True,confirmed
...,...,...,...,...
65,RD_gCC,diffusion_brain_rd,True,confirmed
66,RD_mLEM,diffusion_brain_rd,True,confirmed
67,RD_sCC,diffusion_brain_rd,True,confirmed
68,sFA_c3c5,diffusion_spinal,True,confirmed


Top per-feature longitudinal Cohen d_z ranks used as selector relevance


,rank_by_abs_mean_annual_dz,feature,dz_v1_v2,dz_v2_v3,mean_annual_dz,abs_mean_annual_dz,annual_interval_gap
0,1,Cerebellum_Cortex_CerebNet,-0.877971,-0.403101,-0.640536,0.640536,0.474871
1,2,Pons,-0.777246,-0.359113,-0.568179,0.568179,0.418134
2,3,Cerebellum_WM_CerebNet,-0.510012,-0.342412,-0.426212,0.426212,0.167600
3,4,TotalBrainGMVol_nocereb,-0.402182,-0.422809,-0.412496,0.412496,0.020628
4,5,Lateral_Ventricle,0.535869,0.286104,0.410986,0.410986,0.249765
5,6,Thalamus,-0.469442,-0.248173,-0.358808,0.358808,0.221269
6,7,Midbrain,-0.387709,-0.323785,-0.355747,0.355747,0.063924
7,8,Medulla,-0.544670,-0.127277,-0.335973,0.335973,0.417393
8,9,Putamen,-0.421866,-0.239945,-0.330906,0.330906,0.181922
9,10,Caudate,-0.223088,-0.296276,-0.259682,0.259682,0.073189


Named supervisor panel audit


,panel,n_features,n_present,n_missing,missing_features,features
0,structural_cerebellum_brainstem,6,6,0,,"Cerebellum_WM_CerebNet, Cerebellum_Cortex_Cere..."
1,structural_cerebrum,7,7,0,,"TotalBrainGMVol_nocereb, TotalBrainWMVol_nocer..."
2,structural_spinal,1,1,0,,sCSA_C12_UMN
3,diffusion_cerebellum_brainstem,12,12,0,,"FA_SCP, FA_MCP, FA_ICP, FA_CP, FA_mLEM, FA_PCT..."
4,diffusion_projection,16,16,0,,"FA_ACR, FA_SCR, FA_PCR, FA_ALIC, FA_PLIC, FA_R..."
5,diffusion_association,18,18,0,,"FA_Cing, FA_Cing_h, FA_EC, FA_Fx, FA_Fx_ST, FA..."
6,diffusion_commissural,8,8,0,,"FA_bCC, FA_gCC, FA_sCC, FA_Tap, RD_bCC, RD_gCC..."
7,diffusion_spinal,2,2,0,,"sFA_c3c5, sRD_c3c5"
8,brain_structural,13,13,0,,"Cerebellum_WM_CerebNet, Cerebellum_Cortex_Cere..."
9,spinal_structural,1,1,0,,sCSA_C12_UMN


Spinal structural feature requiring PG/supervisor confirmation


,feature,present,used_in_a_priori_70
0,bt1CSA_C12_UMN,True,False
1,bt2CSA_C12_UMN,True,False
2,sCSA_C12_UMN,True,True


In [12]:
# 70 panel-combination screens:
# 1) anatomical groups and 2) modality/metric groups.
PANEL_COMBO_MAX_SIZE = None  # None means evaluate all non-empty panel combinations.
panel_combo_eval = evaluate_srm_panel_combinations(
    long_df,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    n_boot=N_BOOT,
    max_panel_combo_size=PANEL_COMBO_MAX_SIZE,
    panel_family="anatomical",
)
panel_combo_summary = panel_combo_eval["summary"]
panel_combo_summary.to_csv(RESULTS_DIR / "feature_selection_anatomical_panel_combinations.csv", index=False)

modality_panel_combo_eval = evaluate_srm_panel_combinations(
    long_df,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    n_boot=N_BOOT,
    max_panel_combo_size=PANEL_COMBO_MAX_SIZE,
    panel_family="modality",
)
modality_panel_combo_summary = modality_panel_combo_eval["summary"]
modality_panel_combo_summary.to_csv(RESULTS_DIR / "feature_selection_modality_panel_combinations.csv", index=False)

combined_panel_combo_summary = pd.concat([panel_combo_summary, modality_panel_combo_summary], ignore_index=True)
combined_panel_combo_summary = combined_panel_combo_summary.sort_values(
    ["test_score", "test_annual_interval_gap", "feature_count"],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)
combined_panel_combo_summary["combined_rank"] = np.arange(1, len(combined_panel_combo_summary) + 1)
combined_panel_combo_summary.to_csv(RESULTS_DIR / "feature_selection_panel_combination_summary.csv", index=False)
print("Top panel-combination candidates across anatomical and modality/metric groupings")
display(combined_panel_combo_summary[[
    "combined_rank", "panel_family", "panel_combo", "panel_count", "feature_count",
    "validation_score", "test_score", "validation_minus_test",
    "test_dz_v1_v2", "test_dz_v2_v3", "test_annual_interval_gap", "test_p_progression",
]].head(15))

best_panel_row = combined_panel_combo_summary.iloc[0]
source_results = panel_combo_eval["results"] if best_panel_row["panel_family"] == "anatomical" else modality_panel_combo_eval["results"]
best_panel_features = source_results[best_panel_row["panel_combo"]]["features"]
best_panel_feature_table = pd.DataFrame({"feature": best_panel_features})
best_panel_feature_table.to_csv(RESULTS_DIR / "feature_selection_best_panel_features.csv", index=False)
print("Best a priori panel combination")
display(pd.DataFrame([best_panel_row])[[
    "combined_rank", "panel_family", "panel_combo", "panel_count", "feature_count",
    "validation_score", "test_score", "validation_minus_test",
    "test_dz_v1_v2", "test_dz_v2_v3", "test_annual_interval_gap",
]])

best_panel_fit = fit_locked_srm_full_data(
    long_df,
    best_panel_features,
    pair_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    selection_method="none",
    k=len(best_panel_features),
)
best_panel_importance = coefficient_importance_table(
    best_panel_fit["feature_names"],
    best_panel_fit["coef"],
)
best_panel_importance.to_csv(RESULTS_DIR / "feature_selection_best_panel_importance.csv", index=False)
print("Top feature weights for best a priori panel-combination model")
display(best_panel_importance.head(15))

_, best_panel_contributions = annual_feature_contributions(best_panel_fit, pair_col=subject_col, visit_col="visit")
best_panel_domain_map = infer_feature_domains(best_panel_fit["feature_names"], groups)
best_panel_contribution_table = best_panel_contributions.merge(best_panel_domain_map, on="feature", how="left")
best_panel_contribution_table.to_csv(RESULTS_DIR / "feature_selection_best_panel_contributions.csv", index=False)
print("Top annual feature contributions for best a priori panel-combination model")
display(best_panel_contribution_table.head(15))


Top panel-combination candidates across anatomical and modality/metric groupings


,combined_rank,panel_family,panel_combo,panel_count,feature_count,validation_score,test_score,validation_minus_test,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap,test_p_progression
0,1,modality,brain_structural+brain_diffusion_rd,2,40,0.665371,0.808814,-0.143443,1.000076,0.617553,0.382523,0.813131
1,2,modality,brain_structural+brain_diffusion_rd+spinal_dif...,3,41,0.665801,0.806692,-0.140891,1.004893,0.608490,0.396403,0.823232
2,3,modality,brain_structural+brain_diffusion_rd+spinal_dif...,3,41,0.673005,0.804808,-0.131803,1.003624,0.605991,0.397633,0.818603
3,4,modality,brain_structural+spinal_structural+brain_diffu...,3,41,0.643928,0.803921,-0.159993,1.000286,0.607556,0.392730,0.803030
4,5,modality,brain_structural+brain_diffusion_rd+spinal_dif...,4,42,0.664245,0.803485,-0.139240,0.998894,0.608076,0.390818,0.813552
5,6,modality,brain_structural+spinal_structural+brain_diffu...,4,42,0.643699,0.801421,-0.157722,1.001957,0.600886,0.401071,0.803030
6,7,modality,brain_structural+spinal_structural+brain_diffu...,4,42,0.651526,0.798758,-0.147232,0.999659,0.597857,0.401802,0.813552
7,8,modality,brain_structural+spinal_structural+brain_diffu...,5,43,0.640501,0.797665,-0.157164,0.994857,0.600474,0.394383,0.808081
8,9,anatomical,structural_cerebellum_brainstem+structural_cer...,4,27,0.638421,0.766322,-0.127901,0.953737,0.578907,0.374830,0.800505
9,10,modality,brain_structural+spinal_diffusion_fa,2,14,0.694067,0.764460,-0.070393,0.932588,0.596331,0.336257,0.827862


Best a priori panel combination


,combined_rank,panel_family,panel_combo,panel_count,feature_count,validation_score,test_score,validation_minus_test,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap
0,1,modality,brain_structural+brain_diffusion_rd,2,40,0.665371,0.808814,-0.143443,1.000076,0.617553,0.382523


Top feature weights for best a priori panel-combination model


,feature,standardised_coefficient,absolute_coefficient,coefficient_sign,rank_by_absolute_coefficient
0,Lateral_Ventricle,5.372669,5.372669,1,1
1,Midbrain,-3.961428,3.961428,-1,2
2,Cerebellum_Cortex_CerebNet,-3.446752,3.446752,-1,3
3,Cerebellum_WM_CerebNet,-3.023043,3.023043,-1,4
4,TotalBrainWMVol_nocereb,2.660932,2.660932,1,5
5,Caudate,-2.557411,2.557411,-1,6
6,RD_SCR,2.290592,2.290592,1,7
7,Medulla,2.213929,2.213929,1,8
8,Putamen,2.080574,2.080574,1,9
9,RD_PTR,1.926640,1.926640,1,10


Top annual feature contributions for best a priori panel-combination model


,feature,mean_standardized_feature_change_V1->V2,mean_contribution_V1->V2,mean_standardized_feature_change_V2->V3,mean_contribution_V2->V3,contribution_gap,absolute_annual_contribution,domain
0,Cerebellum_Cortex_CerebNet,-0.146079,0.503497,-0.078906,0.271970,0.231526,0.387734,Other MRI
1,Midbrain,-0.056597,0.224205,-0.059323,0.235004,-0.010800,0.229605,Other MRI
2,Cerebellum_WM_CerebNet,-0.082390,0.249069,-0.061451,0.185768,0.063301,0.217419,Other MRI
3,Lateral_Ventricle,0.039538,0.212422,0.031291,0.168114,0.044308,0.190268,Other MRI
4,Medulla,-0.125666,-0.278215,-0.031784,-0.070368,-0.207847,0.174292,Other MRI
5,SCP,-0.179437,0.223594,-0.051945,0.064729,0.158866,0.144162,Other MRI
6,RD_SCP,0.061410,0.065947,0.159265,0.171032,-0.105085,0.118489,SCP diffusion
7,TotalBrainGMVol_nocereb,-0.080096,0.094093,-0.080958,0.095105,-0.001012,0.094599,Brain morphometry
8,Putamen,-0.053560,-0.111436,-0.026580,-0.055301,-0.056136,0.083369,Other MRI
9,Pons,-0.084069,-0.103538,-0.046040,-0.056702,-0.046836,0.080120,Other MRI


## 2. Feature-Count Elbow Selection


In [13]:
elbow_eval = evaluate_elbow_feature_counts(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    n_boot=N_BOOT,
    k_values=ELBOW_K_VALUES,
    mrmr_lambdas=(0.25, 0.50),
)
elbow_summary = elbow_eval["summary"]
elbow_choice = select_feature_count_elbow(
    elbow_summary,
    performance_tolerance=ELBOW_TOLERANCE,
)
elbow_feature_count_summary = elbow_choice["review_table"]
elbow_summary.to_csv(RESULTS_DIR / "feature_selection_elbow_candidates.csv", index=False)
elbow_feature_count_summary.to_csv(RESULTS_DIR / "feature_selection_elbow_summary.csv", index=False)
print(elbow_choice["summary"])
print("Elbow candidate table: validation/test-style participant-CV screen")
display(elbow_feature_count_summary[[
    "label", "selection_method", "feature_count", "mean_annual_d_z",
    "score_loss_vs_best", "within_elbow_tolerance", "elbow_selected",
    "d_z_v1_v2", "d_z_v2_v3", "annual_interval_gap", "p_progression", "mean_jaccard",
]])
elbow_decision_table = pd.DataFrame([
    {"decision": "raw_best", **elbow_choice["best"]},
    {"decision": "elbow_selected", **elbow_choice["chosen"]},
])
print("Elbow decision summary")
display(elbow_decision_table)


Elbow-selected 16 features (progression_univariate_k16) because its mean_annual_d_z=0.747 is within 0.030 of the best mean_annual_d_z=0.767.
Elbow candidate table: validation/test-style participant-CV screen


,label,selection_method,feature_count,mean_annual_d_z,score_loss_vs_best,within_elbow_tolerance,elbow_selected,d_z_v1_v2,d_z_v2_v3,annual_interval_gap,p_progression,mean_jaccard
0,progression_mrmr_k4_lam0.5,progression_mrmr,4,0.662917,0.103780,False,False,0.860172,0.465663,0.394509,0.754630,0.840000
1,progression_mrmr_k4_lam0.25,progression_mrmr,4,0.628424,0.138274,False,False,0.821049,0.435798,0.385251,0.740320,0.720000
2,progression_univariate_k4,progression_univariate,4,0.619351,0.147346,False,False,0.804911,0.433791,0.371120,0.729798,0.626667
3,progression_univariate_k6,progression_univariate,6,0.685279,0.081418,False,False,0.857400,0.513158,0.344243,0.755051,0.735714
4,progression_mrmr_k6_lam0.5,progression_mrmr,6,0.679126,0.087572,False,False,0.831142,0.527109,0.304033,0.765572,0.800000
5,progression_mrmr_k6_lam0.25,progression_mrmr,6,0.658343,0.108354,False,False,0.791666,0.525020,0.266646,0.742003,0.678571
6,progression_univariate_k8,progression_univariate,8,0.705009,0.061688,False,False,0.871395,0.538623,0.332772,0.754209,0.822222
7,progression_mrmr_k8_lam0.25,progression_mrmr,8,0.687272,0.079425,False,False,0.815351,0.559194,0.256157,0.745370,0.742222
8,progression_mrmr_k8_lam0.5,progression_mrmr,8,0.678728,0.087969,False,False,0.826059,0.531397,0.294662,0.761785,0.714343
9,progression_univariate_k10,progression_univariate,10,0.735359,0.031338,False,False,0.923947,0.546771,0.377176,0.773990,0.854545


Elbow decision summary


,decision,label,selection_method,k,feature_count,mean_annual_d_z,d_z_v1_v2,d_z_v2_v3,annual_interval_gap,p_progression,pooled_pair_d_z,n_subjects,cv_mode,cv_n_splits,mean_jaccard,selection_params,rank
0,raw_best,progression_univariate_k40,progression_univariate,40,40,0.766697,0.911280,0.622114,0.289166,0.794192,0.764926,207,group_kfold,5,0.694148,{},1
1,elbow_selected,progression_univariate_k16,progression_univariate,16,16,0.747151,0.851383,0.642919,0.208464,0.813973,0.739833,207,group_kfold,5,0.790437,{},4


## 3. Selector Method Comparison


In [14]:
selection_candidates = elbow_candidate_grid(
    k_values=ELBOW_K_VALUES,
    include_full_count=len(imaging_cols),
    mrmr_lambdas=(0.25, 0.50),
) + [
    {"label": "a_priori_70_sparse_srm_lam0.01_alpha0.5", "selection_method": "sparse_srm", "k": int(elbow_choice["chosen"].get("k", 8)), "selection_params": {"sparse_lambda": 0.01, "sparse_alpha": 0.5}},
    {"label": "a_priori_70_mml", "selection_method": "mml", "k": int(elbow_choice["chosen"].get("k", 8)), "selection_params": {}},
    {"label": "a_priori_70_mi_visit", "selection_method": "mi_visit", "k": int(elbow_choice["chosen"].get("k", 8)), "selection_params": {}},
]

results = []
selected = {}
interval_results = {}
for cand in selection_candidates:
    res = srm_global_loocv(
        long_df,
        imaging_cols,
        subject_col=subject_col,
        visit_col="visit",
        selection_method=cand["selection_method"],
        k=int(cand["k"]),
        cv_n_splits=CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
        selection_params=cand.get("selection_params", {}),
        compute_ci=False,
    )
    label = cand["label"]
    selected[label] = res["selected_features_by_fold"]
    annual_intervals = adjacent_pair_interval_effect_summary(
        res["oof_df"],
        pair_col=subject_col,
        visit_col="visit",
        score_col="score",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    interval_results[label] = annual_intervals
    annual_diag = annual_tuning_diagnostics(annual_intervals)
    pooled_deltas = paired_deltas_from_long(res["oof_df"].rename(columns={"score": "value"}), subject_col, "visit", "value")
    stability = feature_set_jaccard_summary(selected[label])
    median_features = int(np.median([len(x) for x in selected[label]])) if selected[label] else 0
    representative_features = selected[label][0] if selected[label] else []
    domains = feature_domain_coverage(representative_features, FEATURE_GROUPS)
    results.append({
        "label": label,
        "selection_method": cand["selection_method"],
        "k/features": cand["k"],
        "selection_params": cand.get("selection_params", {}),
        "pooled_pair_d_z_reference": res["d_score"],
        **annual_diag,
        "pooled_p_progression_reference": probability_positive_change(pooled_deltas),
        "n_subject_pairs": res["n_subjects"],
        "feature_count": median_features,
        "mean_jaccard": stability["mean_jaccard"],
        "median_jaccard": stability["median_jaccard"],
        "iqr_jaccard": stability["iqr_jaccard"],
        "min_jaccard": stability["min_jaccard"],
        "max_jaccard": stability["max_jaccard"],
        "domains": domains.get("represented_domains", ""),
    })

selection_results = pd.DataFrame(results).sort_values(
    ["mean_validation_annual_dz", "annual_interval_gap"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)
selection_results.to_csv(RESULTS_DIR / "feature_selection_method_comparison.csv", index=False)
print("Top a priori 70 feature-selection candidates")
display(selection_results[[
    "label", "selection_method", "k/features", "feature_count",
    "mean_validation_annual_dz", "dz_v1_v2", "dz_v2_v3",
    "annual_interval_gap", "pooled_p_progression_reference", "mean_jaccard",
]].head(12))


Top a priori 70 feature-selection candidates


,label,selection_method,k/features,feature_count,mean_validation_annual_dz,dz_v1_v2,dz_v2_v3,annual_interval_gap,pooled_p_progression_reference,mean_jaccard
0,progression_univariate_k40,progression_univariate,40,40,0.766697,0.911280,0.622114,0.289166,0.797101,0.694148
1,progression_mrmr_k40_lam0.25,progression_mrmr,40,40,0.762384,0.893434,0.631334,0.262100,0.792271,0.696695
2,progression_mrmr_k40_lam0.5,progression_mrmr,40,40,0.757296,0.964750,0.549842,0.414908,0.792271,0.706899
3,progression_univariate_k16,progression_univariate,16,16,0.747151,0.851383,0.642919,0.208464,0.816425,0.790437
4,progression_univariate_k10,progression_univariate,10,10,0.735359,0.923947,0.546771,0.377176,0.777778,0.854545
5,progression_univariate_k20,progression_univariate,20,20,0.733410,0.843791,0.623029,0.220762,0.792271,0.710017
6,progression_univariate_k12,progression_univariate,12,12,0.723842,0.852975,0.594709,0.258266,0.792271,0.938462
7,progression_mrmr_k16_lam0.25,progression_mrmr,16,16,0.722610,0.803115,0.642104,0.161012,0.782609,0.733031
8,progression_mrmr_k20_lam0.5,progression_mrmr,20,20,0.721190,0.806958,0.635421,0.171536,0.777778,0.664003
9,progression_mrmr_k10_lam0.25,progression_mrmr,10,10,0.720720,0.903627,0.537814,0.365813,0.806763,0.854545


## 4. Parsimony And Stability Decision


In [15]:
review_input = selection_results.rename(columns={"label": "param_selection_label"}).copy()
review_input["param_selection_method"] = review_input["selection_method"]
review_input["se_validation_dz"] = selection_results["mean_validation_annual_dz"].std(ddof=1) / np.sqrt(max(len(selection_results), 1))
review_input["jaccard_stability"] = review_input["mean_jaccard"]
review_input["sign_stability"] = np.nan
review_input["score_ranking_stability"] = np.nan
selector_review = tuning_recommendation(review_input)

simplest_near_optimal = selector_review["near_optimal"].sort_values(
    ["feature_count", "annual_interval_gap"], ascending=[True, True]
).head(1)
most_stable_near_optimal = selector_review["near_optimal"].sort_values(
    ["jaccard_stability", "feature_count"], ascending=[False, True]
).head(1)
selector_decision_table = pd.concat([
    pd.DataFrame([{ "decision": "raw_best", **selector_review["raw_best"] }]),
    simplest_near_optimal.assign(decision="simplest_near_optimal"),
    most_stable_near_optimal.assign(decision="most_stable_near_optimal"),
    pd.DataFrame([{ "decision": "recommended", **selector_review["recommended"] }]),
], ignore_index=True, sort=False)
selector_decision_table.to_csv(RESULTS_DIR / "feature_selection_selector_decision_summary.csv", index=False)
print("Feature-selection decision summary; near-optimal candidate count:", len(selector_review["near_optimal"]))
display(selector_decision_table[[
    "decision", "param_selection_label", "param_selection_method", "feature_count",
    "mean_validation_annual_dz", "annual_interval_gap", "jaccard_stability",
]])
print(selector_review["summary"])
print("Human-verification summary")
display(tuning_verification_summary(selector_review))


Feature-selection decision summary; near-optimal candidate count: 4


,decision,param_selection_label,param_selection_method,feature_count,mean_validation_annual_dz,annual_interval_gap,jaccard_stability
0,raw_best,progression_univariate_k40,progression_univariate,40,0.766697,0.289166,0.694148
1,simplest_near_optimal,progression_univariate_k16,progression_univariate,16,0.747151,0.208464,0.790437
2,most_stable_near_optimal,progression_univariate_k16,progression_univariate,16,0.747151,0.208464,0.790437
3,recommended,progression_univariate_k16,progression_univariate,16,0.747151,0.208464,0.790437


Recommended candidate is within one SE of the raw best (performance difference 0.01955) and is preferred by the implemented hierarchy: smaller annual interval gap first, then higher P(delta>0), fewer features, and available stability diagnostics.
Human-verification summary


,item,value
0,Best raw-performance parameters,{'selection_label': 'progression_univariate_k4...
1,Recommended parameters,{'selection_label': 'progression_univariate_k1...
2,Difference in performance,0.019546
3,Reason for recommendation,Recommended candidate is within one SE of the ...
4,Any instability/warning,No automatic warning.


## 5. Nested Inner-CV Confirmation


In [16]:
nested_candidates = elbow_nested_candidates(
    elbow_choice["chosen"],
    ridge_grid=(0.0,),
    covariance_shrinkage_grid=(0.0, 0.45),
    z_clip_grid=(None,)
)
# Keep the full 70-feature panel as an explicit reference candidate in nested tuning.
nested_candidates.extend([
    {"ridge": 0.0, "covariance_shrinkage": 0.45, "z_clip": None, "selection_method": "none", "k": len(imaging_cols), "selection_label": "a_priori_70_all_reference"},
])

pd.DataFrame(nested_candidates).to_csv(RESULTS_DIR / "feature_selection_elbow_nested_candidates.csv", index=False)
nested_res = srm_global_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=nested_candidates,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=False,
    tuning_metric="annual_mean_dz",
)

nested_intervals = adjacent_pair_interval_effect_summary(
    nested_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
nested_diag = annual_tuning_diagnostics(nested_intervals)
nested_stability = feature_set_jaccard_summary(nested_res["selected_features_by_fold"])
nested_res["chosen_params_df"].to_csv(RESULTS_DIR / "feature_selection_elbow_nested_chosen_params.csv", index=False)
nested_intervals.to_csv(RESULTS_DIR / "feature_selection_elbow_nested_intervals.csv", index=False)
nested_summary = pd.DataFrame([{**nested_diag, **nested_stability, "outer_cv_d_score": nested_res["d_score"]}])
nested_summary.to_csv(RESULTS_DIR / "feature_selection_elbow_nested_summary.csv", index=False)
chosen_selector_counts = nested_res["chosen_params_df"].get("selection_label", pd.Series(dtype=object)).value_counts().rename_axis("selection_label").reset_index(name="n_outer_folds")
chosen_selector_counts.to_csv(RESULTS_DIR / "feature_selection_elbow_nested_selector_counts.csv", index=False)
print("Nested selector tuning diagnostics")
display(nested_summary)
print("Nested annual OOF interval performance")
display(nested_intervals)
print("Chosen selector counts across outer folds")
display(chosen_selector_counts)


Nested selector tuning diagnostics


,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,mean_jaccard,median_jaccard,iqr_jaccard,min_jaccard,max_jaccard,outer_cv_d_score
0,0.837615,0.611331,0.724473,0.226284,0.779461,0.471577,0.228571,0.525815,0.228571,1.0,0.719325


Nested annual OOF interval performance


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,1.113123,1.328919,0.837615,0.668478,1.051475,0.851852
1,V2->V3,99,0.945009,1.545822,0.611331,0.420451,0.815450,0.707071


Chosen selector counts across outer folds


,selection_label,n_outer_folds
0,progression_univariate_k16,3
1,a_priori_70_all_reference,2


## 6. Full-Feature Reference Check


In [17]:
none_row = selection_results[selection_results["selection_method"].eq("none")].head(1)
if len(none_row):
    ref = none_row.iloc[0]
    comparison = selection_results.copy()
    comparison["delta_mean_annual_dz_vs_none"] = comparison["mean_validation_annual_dz"] - ref["mean_validation_annual_dz"]
    comparison["delta_d12_vs_none"] = comparison["dz_v1_v2"] - ref["dz_v1_v2"]
    comparison["delta_d23_vs_none"] = comparison["dz_v2_v3"] - ref["dz_v2_v3"]
    comparison["features_removed_vs_none"] = ref["feature_count"] - comparison["feature_count"]
    comparison["percent_feature_reduction_vs_none"] = 100.0 * comparison["features_removed_vs_none"] / max(float(ref["feature_count"]), 1.0)
    comparison["stability_difference_vs_none"] = comparison["mean_jaccard"] - ref["mean_jaccard"]
    comparison.to_csv(RESULTS_DIR / "feature_selection_vs_full_feature_reference.csv", index=False)
    print("Largest changes versus full 70-feature reference")
    display(comparison[[
        "label", "selection_method", "delta_mean_annual_dz_vs_none", "delta_d12_vs_none",
        "delta_d23_vs_none", "features_removed_vs_none", "percent_feature_reduction_vs_none",
        "stability_difference_vs_none",
    ]].head(12))
else:
    print("No full-feature reference row found.")


Largest changes versus full 70-feature reference


,label,selection_method,delta_mean_annual_dz_vs_none,delta_d12_vs_none,delta_d23_vs_none,features_removed_vs_none,percent_feature_reduction_vs_none,stability_difference_vs_none
0,progression_univariate_k40,progression_univariate,0.109149,0.105622,0.112675,30,42.857143,-0.305852
1,progression_mrmr_k40_lam0.25,progression_mrmr,0.104836,0.087776,0.121895,30,42.857143,-0.303305
2,progression_mrmr_k40_lam0.5,progression_mrmr,0.099747,0.159092,0.040403,30,42.857143,-0.293101
3,progression_univariate_k16,progression_univariate,0.089602,0.045724,0.133480,54,77.142857,-0.209563
4,progression_univariate_k10,progression_univariate,0.077811,0.118289,0.037333,60,85.714286,-0.145455
5,progression_univariate_k20,progression_univariate,0.075862,0.038133,0.113591,50,71.428571,-0.289983
6,progression_univariate_k12,progression_univariate,0.066294,0.047317,0.085270,58,82.857143,-0.061538
7,progression_mrmr_k16_lam0.25,progression_mrmr,0.065061,-0.002543,0.132665,54,77.142857,-0.266969
8,progression_mrmr_k20_lam0.5,progression_mrmr,0.063641,0.001299,0.125983,50,71.428571,-0.335997
9,progression_mrmr_k10_lam0.25,progression_mrmr,0.063172,0.097968,0.028375,60,85.714286,-0.145455


## 7. Feature Stability And Interpretation


In [18]:
stability_tables = {}
for label, folds in selected.items():
    table = feature_stability_report(folds, imaging_cols).copy()
    stability_tables[label] = table

freq_parts = []
for label, table in stability_tables.items():
    freq_parts.append(table[["feature", "selection_frequency"]].rename(columns={"selection_frequency": label}))
feature_frequency = freq_parts[0]
for part in freq_parts[1:]:
    feature_frequency = feature_frequency.merge(part, on="feature", how="outer")
method_cols = [c for c in feature_frequency.columns if c != "feature"]
feature_frequency["max_selection_frequency"] = feature_frequency[method_cols].max(axis=1)
feature_frequency = feature_frequency.sort_values("max_selection_frequency", ascending=False, kind="mergesort")
feature_frequency.to_csv(RESULTS_DIR / "feature_selection_feature_frequency.csv", index=False)
print("Most repeatedly selected features across selectors")
display(feature_frequency.head(25))

selected_feature_rows = []
for label, folds in selected.items():
    table = stability_tables[label]
    chosen = table[table["selection_frequency"] > 0].copy()
    for _, row in chosen.sort_values("selection_frequency", ascending=False).iterrows():
        selected_feature_rows.append({
            "label": label,
            "feature": row["feature"],
            "selection_frequency": row["selection_frequency"],
            "n_selections": row["n_selections"],
        })
selected_feature_table = pd.DataFrame(selected_feature_rows)
selected_feature_table.to_csv(RESULTS_DIR / "feature_selection_selected_features_by_configuration.csv", index=False)
selected_feature_summary = selected_feature_table.sort_values(
    ["selection_frequency", "n_selections"], ascending=[False, False]
).head(30)
print("Top selected features by configuration")
display(selected_feature_summary)


Most repeatedly selected features across selectors


,feature,progression_univariate_k4,progression_mrmr_k4_lam0.25,progression_mrmr_k4_lam0.5,progression_univariate_k6,progression_mrmr_k6_lam0.25,progression_mrmr_k6_lam0.5,progression_univariate_k8,progression_mrmr_k8_lam0.25,progression_mrmr_k8_lam0.5,...,progression_mrmr_k30_lam0.25,progression_mrmr_k30_lam0.5,progression_univariate_k40,progression_mrmr_k40_lam0.25,progression_mrmr_k40_lam0.5,all_70,a_priori_70_sparse_srm_lam0.01_alpha0.5,a_priori_70_mml,a_priori_70_mi_visit,max_selection_frequency
0,Caudate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
1,Cerebellum_Cortex_CerebNet,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.4,1.0
2,Cerebellum_WM_CerebNet,0.8,0.2,0.2,0.8,0.8,0.6,1.0,1.0,0.8,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.2,1.0
3,FA_ACR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.4,0.2,0.4,1.0,0.2,0.0,0.0,1.0
4,FA_ALIC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.2,0.0,0.0,1.0,0.8,0.0,0.2,1.0
5,FA_CP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.4,0.4,0.6,0.8,1.0,1.0,0.0,0.2,1.0
6,FA_CST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.2,0.0,0.2,1.0
7,FA_Cing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
8,FA_Cing_h,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.6,0.0,0.0,1.0
9,FA_EC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.2,0.0,0.0,1.0


Top selected features by configuration


,label,feature,selection_frequency,n_selections
0,progression_univariate_k4,Cerebellum_Cortex_CerebNet,1.0,5
1,progression_univariate_k4,Pons,1.0,5
6,progression_mrmr_k4_lam0.25,Cerebellum_Cortex_CerebNet,1.0,5
7,progression_mrmr_k4_lam0.25,Pons,1.0,5
8,progression_mrmr_k4_lam0.25,Lateral_Ventricle,1.0,5
12,progression_mrmr_k4_lam0.5,Cerebellum_Cortex_CerebNet,1.0,5
13,progression_mrmr_k4_lam0.5,Pons,1.0,5
14,progression_mrmr_k4_lam0.5,Lateral_Ventricle,1.0,5
17,progression_univariate_k6,Cerebellum_Cortex_CerebNet,1.0,5
18,progression_univariate_k6,Pons,1.0,5


## 8. Fold-Aware Benchmarks


In [19]:
# Fold-level clinical train/test benchmark for supervisor review.
weekly_fold_train_test_clinical_benchmarks = fold_train_test_clinical_benchmark_table(
    long_df,
    pairs_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    clinical_scales=("FARS", "SARA"),
    pair_types=("V1V2", "V2V3"),
)
weekly_fold_train_test_clinical_benchmarks.to_csv(
    RESULTS_DIR / "weekly_fold_train_test_clinical_benchmarks.csv",
    index=False,
)
clinical_display_cols = [
    "fold",
    "clinical_scale",
    "train_n_subjects",
    "test_n_subjects",
    "clinical_train_n_pairs",
    "clinical_test_n_pairs",
    "clinical_train_d",
    "clinical_test_d",
    "clinical_train_minus_test_d",
]
print("Feature selection fold-level train/test clinical benchmarks")
display(weekly_fold_train_test_clinical_benchmarks[clinical_display_cols])


Feature selection fold-level train/test clinical benchmarks


,fold,clinical_scale,train_n_subjects,test_n_subjects,clinical_train_n_pairs,clinical_test_n_pairs,clinical_train_d,clinical_test_d,clinical_train_minus_test_d
0,1,FARS,93,24,162,45,0.361604,0.585453,-0.223849
1,1,SARA,93,24,162,45,0.379539,0.503772,-0.124232
2,2,FARS,93,24,159,48,0.384278,0.481659,-0.097381
3,2,SARA,93,24,159,48,0.357044,0.553715,-0.196670
4,3,FARS,94,23,169,38,0.424238,0.332809,0.091429
5,3,SARA,94,23,169,38,0.407784,0.402383,0.005401
6,4,FARS,94,23,167,40,0.406111,0.421720,-0.015609
7,4,SARA,94,23,167,40,0.458035,0.196119,0.261916
8,5,FARS,94,23,171,36,0.460649,0.213516,0.247133
9,5,SARA,94,23,171,36,0.421557,0.332821,0.088736
